# 1T Feature-Family Classifier Contracts

This notebook compares feature-family classifier scores by replaying each score column into the standard strategy contract. Each output directory contains `trade_list.parquet`, which is the only input the options backtester needs for option-equivalent trade simulation.

In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
from quant_orchestrator.artifact_contracts import read_strategy_artifacts
if REPO_ROOT.name != 'quant-orchestrator':
    REPO_ROOT = next(parent for parent in REPO_ROOT.parents if parent.name == 'quant-orchestrator')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_orchestrator.platforms.backtesting_frameworks import (
    ScoredPanelTopKReplayConfig,
    replay_scored_panel_top_k,
)

pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 240)
print('repo_root', REPO_ROOT)

## Configuration

In [ ]:
SCORED_PANEL_PATH = None  # Example: REPO_ROOT / 'artifacts/path/to/scored_panel.parquet'
OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'classifier_1t_feature_family_contracts'

# Explicit score columns are preferred. If empty, the notebook infers conservative score-like columns.
SCORE_COLUMNS = []
FALLBACK_SCORE_COLUMNS = ('prob_buy', 'ensemble_mean')
MAX_SCORE_COLUMNS = 0  # 0 means no cap.

TOP_K = 5
THRESHOLD = 0.50
INITIAL_BALANCE = 100_000.0
FEE_BPS = 5.0
SLIPPAGE_BPS = 5.0

print('output_root', OUTPUT_ROOT)

## Load Scored Panel

In [ ]:
if SCORED_PANEL_PATH is not None:
    scored_panel = pd.read_parquet(SCORED_PANEL_PATH)
elif 'scored_panel' not in globals():
    raise RuntimeError('Provide SCORED_PANEL_PATH or define scored_panel before running this notebook.')

scored_panel = scored_panel.copy()
scored_panel['date'] = pd.to_datetime(scored_panel['date'])
scored_panel['symbol'] = scored_panel['symbol'].astype(str).str.upper()
display(scored_panel.head())
print('rows', len(scored_panel), 'symbols', scored_panel['symbol'].nunique())

## Resolve Score Columns

In [ ]:
required = {'date', 'symbol', 'close'}
missing = sorted(required.difference(scored_panel.columns))
if missing:
    raise RuntimeError(f'scored_panel is missing required columns: {missing}')

if SCORE_COLUMNS:
    score_columns = list(SCORE_COLUMNS)
else:
    inferred = []
    for col in scored_panel.columns:
        name = str(col)
        if name in required:
            continue
        if name in FALLBACK_SCORE_COLUMNS or name.endswith('__prob_buy') or name.endswith('_prob_buy') or name.endswith('_score'):
            inferred.append(name)
    score_columns = sorted(dict.fromkeys(inferred))

if MAX_SCORE_COLUMNS > 0:
    score_columns = score_columns[:MAX_SCORE_COLUMNS]
missing_scores = [col for col in score_columns if col not in scored_panel.columns]
if missing_scores:
    raise RuntimeError(f'SCORE_COLUMNS not found in scored_panel: {missing_scores}')
if not score_columns:
    raise RuntimeError('No score columns found. Set SCORE_COLUMNS explicitly.')

print('score_columns', len(score_columns))
display(pd.DataFrame({'score_col': score_columns}))

## Replay Each Feature Family

In [ ]:
def safe_name(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value).strip('_') or 'score'


summary_rows = []
contracts = {}
for score_col in score_columns:
    output_dir = OUTPUT_ROOT / safe_name(score_col)
    config = ScoredPanelTopKReplayConfig(
        score_col=score_col,
        top_k=TOP_K,
        threshold=THRESHOLD,
        initial_balance=INITIAL_BALANCE,
        fee_bps=FEE_BPS,
        slippage_bps=SLIPPAGE_BPS,
        strategy_name=f'classifier_1t.feature_family.{score_col}',
        output_dir=output_dir,
        metadata={'notebook': 'classifier_1t_feature_family_option_windows.ipynb', 'score_col': score_col},
    )
    result = replay_scored_panel_top_k(scored_panel, config=config)
    bundle = read_strategy_artifacts(output_dir)
    contracts[score_col] = bundle
    performance = result.summary.get('performance', {})
    summary_rows.append({
        'score_col': score_col,
        'final_equity': performance.get('final_equity'),
        'total_return_pct': performance.get('total_return_pct'),
        'total_return_multiple': performance.get('total_return_multiple'),
        'max_drawdown_pct': performance.get('max_drawdown_pct'),
        'trades': len(result.rule_replay.trade_list),
        'actions': len(result.rule_replay.action_tape),
        'trade_list_path': str(bundle.trade_list_path),
        'manifest_path': str(bundle.manifest_path),
    })

summary = pd.DataFrame(summary_rows).sort_values(['total_return_pct', 'trades'], ascending=[False, False])
display(summary)

## Option Backtest Inputs

In [ ]:
option_inputs = summary[['score_col', 'trade_list_path', 'manifest_path', 'trades']].copy()
display(option_inputs)

if not summary.empty:
    best_score_col = summary.iloc[0]['score_col']
    print('best_score_col', best_score_col)
    print('best_trade_list', contracts[best_score_col].trade_list_path)
    display(pd.read_parquet(contracts[best_score_col].trade_list_path).tail(20))